# **animove**: `<TODO :: title of project here>`

**By**:, Zhean Robby Ganituen, Jaztin Jacob Jimenez, Reece Benedict Orense, Ashley Paulianna Reyes, and Matthew Fraser Sim

---

## **introduction**
**Dataset**: LaPoint, S. 2026. Data from: Study “Carnivore movements near Black Rock Forest New York” [part]. Movebank Data Repository.

**Paper**: Oliver, R.Y. et al. 2026. Interacting effects of human presence and landscape modification on birds and mammals. Science. 392, 6800 (May 2026), 879–884. https://doi.org/10.1126/science.adq3396.

`<TODO :: add dataset description here>`

`<TODO :: add plan with the dataset>`

## **0 setup**

Let's first import the Python libraries needed for the project and define some filenames which will be used throughout the dataset.

In [1]:
# Imports
import numpy as np
import pandas as pd

# Directories and filenames
DATA_DIR = "../data/"
RAW_DATA_DIR = DATA_DIR + "raw/"
PROC_DATA_DIR = DATA_DIR + "processed/"

# DataFrames global variables
MAIN_DF = 0
REF_DF = 0
JOIN_DF = 0

### **0.1 dataset loading**

Now, let's load the two raw datasets as a pandas `DataFrame`.

In [2]:
MAIN_DF= pd.read_csv(RAW_DATA_DIR + "main.csv")
REF_DF = pd.read_csv(RAW_DATA_DIR + "reference.csv")

/tmp/ipykernel_19409/1250527727.py:1: DtypeWarning: Columns (0: manually-marked-outlier) have mixed types. Specify dtype option on import or set low_memory=False.
  MAIN_DF= pd.read_csv(RAW_DATA_DIR + "main.csv")


## **0.2 joining**

Before doing any cleaning, let's first join the two. 

To get an idea of what the dataset looks like. Let's first print the columns and information from each dataframe. From this, we can come up with a methodology for joining the two.

In [3]:
print("\t========== Main DataFrame ========== ")
print(MAIN_DF.info())

	========== Main DataFrame ========== 
<class 'pandas.DataFrame'>
RangeIndex: 131930 entries, 0 to 131929
Data columns (total 26 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   event-id                           131930 non-null  int64  
 1   visible                            131930 non-null  bool   
 2   timestamp                          131930 non-null  str    
 3   location-long                      126483 non-null  float64
 4   location-lat                       126483 non-null  float64
 5   data-decoding-software             131930 non-null  int64  
 6   eobs:battery-voltage               131930 non-null  int64  
 7   eobs:fix-battery-voltage           131930 non-null  int64  
 8   eobs:horizontal-accuracy-estimate  126483 non-null  float64
 9   eobs:key-bin-checksum              131930 non-null  int64  
 10  eobs:speed-accuracy-estimate       126483 non-null  float64
 11  eobs:start-

In [4]:
print("\t========== Reference DataFrame ========== ")
print(REF_DF.info())

	========== Reference DataFrame ========== 
<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   tag-id                 9 non-null      int64  
 1   animal-id              9 non-null      str    
 2   animal-taxon           9 non-null      str    
 3   deploy-on-date         9 non-null      str    
 4   deploy-off-date        8 non-null      str    
 5   animal-life-stage      9 non-null      str    
 6   animal-mass            9 non-null      float64
 7   animal-offspring       1 non-null      str    
 8   animal-parents         2 non-null      str    
 9   animal-sex             9 non-null      str    
 10  animal-siblings        2 non-null      str    
 11  attachment-body-part   9 non-null      str    
 12  attachment-type        9 non-null      str    
 13  deploy-on-latitude     1 non-null      float64
 14  deploy-on-longitude    1 non-

From here, the column `individual-local-identifier` (`ili`) from `MAIN_DF` and `animal-id` from `REF_DF` seem to be related to each other. Let's verify this claim by checking that the set of possible values for each column is the same.

In [5]:
unique_ili_id = set(MAIN_DF["individual-local-identifier"].unique())
unique_animal_id = set(REF_DF["animal-id"].unique())

print("\t========== Is ili and animal-id a potential join key? ==========")
if unique_ili_id == unique_animal_id:
  print("YES")
  print(unique_ili_id)
else:
  print("NO")

	========== Is ili and animal-id a potential join key? ==========
YES
{'LYRU_0002', 'LYRU_0006', 'LYRU_0008', 'PEPE_0001', 'LYRU_0005', 'LYRU_0001', 'PEPE_0002', 'LYRU_0003', 'LYRU_0004'}


Now that we show that `ili` and `animal-id` is the join key. We will now join `MAIN_DF` and `REF_DF` to construct `JOIN_DF`. But let's first rename `ili` to `animal-id`. Then, we'll call `pd.merge` on the join key (which is now `animal-id`).

In [8]:
MAIN_DF = MAIN_DF.rename(columns={"individual-local-identifier": "animal-id"})
JOIN_DF = pd.merge(MAIN_DF, REF_DF, on="animal-id")

Now let's see the columns and information of the dataset after performing the join. Let's ensure that information of the two datasets were kept after merging by performing some tests.

In [ ]:
print("\t========== Joined DataFrame ========== ")
print(JOIN_DF.info())
print(f"""
=== The total number of columns must be len(MAIN_DF.columns) + len(REF_DF.columns) - 1
{len(MAIN_DF.columns) + len(REF_DF.columns) - 1} == {len(JOIN_DF.columns)}
""")
print(f"""
=== The total number of entries must be the max(len(MAIN_DF), len(REF_DF))
{max(len(MAIN_DF), len(REF_DF))} == {len(JOIN_DF)}
""")

	========== Joined DataFrame ========== 
<class 'pandas.DataFrame'>
RangeIndex: 131930 entries, 0 to 131929
Data columns (total 49 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   event-id                           131930 non-null  int64  
 1   visible                            131930 non-null  bool   
 2   timestamp                          131930 non-null  str    
 3   location-long                      126483 non-null  float64
 4   location-lat                       126483 non-null  float64
 5   data-decoding-software             131930 non-null  int64  
 6   eobs:battery-voltage               131930 non-null  int64  
 7   eobs:fix-battery-voltage           131930 non-null  int64  
 8   eobs:horizontal-accuracy-estimate  126483 non-null  float64
 9   eobs:key-bin-checksum              131930 non-null  int64  
 10  eobs:speed-accuracy-estimate       126483 non-null  float64
 11  eobs:star